In [2]:
!pip install mistralai chromadb "psycopg[binary]" -q

from typing import Dict, Any
from mistralai.client import Mistral
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings
from chromadb.utils.embedding_functions import register_embedding_function
import psycopg

In [ ]:
MISTRAL_API_KEY = "XXX"

def MistralEmbeddingAPI(s_list):
    with Mistral(api_key=MISTRAL_API_KEY) as mistral:
        res = mistral.embeddings.create(model="mistral-embed", inputs=s_list)
        # print(res)
        return [x.embedding for x in res.data]

In [4]:
chroma_client = chromadb.PersistentClient(path="./comic_embeddings_db")
collection = chroma_client.get_or_create_collection("comic_embeddings")

In [ ]:
# chroma_client.delete_collection("comic_embeddings")

In [5]:
@register_embedding_function
class MistralEmbeddingFunction(EmbeddingFunction):

    def __init__(self, model):
        self.model = model

    def __call__(self, input: Documents) -> Embeddings:
        
        return MistralEmbeddingAPI(input)

    @staticmethod
    def name() -> str:
        return "my-ef"

    def get_config(self) -> Dict[str, Any]:
        return dict(model=self.model)

    @staticmethod
    def build_from_config(config: Dict[str, Any]) -> "EmbeddingFunction":
        return MyEmbeddingFunction(config['model'])

In [36]:
comic = []
with psycopg.connect("postgresql://postgres@localhost:5432/dev?password=12355") as conn:
    with conn.cursor() as cur:
        cur.execute("select * from comic")
        for row in cur:
            comic.append(row)


In [47]:
comic_documents = []
comic_id = []
for c in comic:
    comic_documents.append(c[4])
    comic_id.append(str(c[0]))

In [48]:
collection.upsert(
    ids=comic_id,
    documents=comic_documents
)

In [6]:
collection.query(
    query_texts=["ghoul"],
    n_results = 5
)

{'ids': [['2258', '2002', '2206', '1953', '2159']],
 'embeddings': None,
 'documents': [['Lurking within the shadows of Tokyo are frightening beings known as "ghouls," who satisfy their hunger by feeding on humans once night falls. Ken Kaneki, an unsuspecting university freshman, finds himself caught in a world between humans and ghouls when his date turns out to be a ghoul after his flesh.',
   '"Berserk" is a dark fantasy manga series created by Kentaro Miura, first published in 1988. The story follows Guts, a tragic antihero known as the Black Swordsman, who battles his way through a brutal and unforgiving world filled with demons, knights, and moral complexities. Guts\' journey is marked by loss and vengeance, as he seeks to save his lover, Casca, who is imprisoned by the demonic figure Femto. Throughout the series, themes of good versus evil, duality, and the darker aspects of human nature are explored, particularly through Guts\' struggles with the Beast of Darkness, a manifestat